In [100]:
import json
import torch
import random
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import get_peft_model, LoraConfig, TaskType
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Fix random seeds for reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

os.makedirs('results', exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Executing on: {device}")

Executing on: cuda


In [101]:
# 1. Load Data
texts, labels = [], []
with open('extended_corpus.jsonl', 'r') as f:
    for line in f:
        data = json.loads(line)
        texts.append(data['text'])
        labels.append(data['label'])

# 2. HuggingFace Tokenizer
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
encoded_data = tokenizer(texts, padding=True, truncation=True, max_length=50, return_tensors='pt')

# 3. Dataset & Split (80/20)
class EmailDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item
    def __len__(self):
        return len(self.labels)

split_idx = int(0.8 * len(labels))
train_dataset = EmailDataset({k: v[:split_idx] for k, v in encoded_data.items()}, labels[:split_idx])
val_dataset = EmailDataset({k: v[split_idx:] for k, v in encoded_data.items()}, labels[split_idx:])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

In [102]:
def evaluate_model(model, dataloader, title, filename):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, zero_division=0)
    rec = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)

    # Plot Confusion Matrix
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Benign', 'Phishing'], yticklabels=['Benign', 'Phishing'])
    plt.title(f'{title} Confusion Matrix')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.savefig(f'results/{filename}.png')
    plt.close()

    return [acc, prec, rec, f1]

In [103]:
# Load Base System
base_model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2).to(device)

print("Evaluating Base System (Frozen DistilBERT + Untrained Head)...")
base_metrics = evaluate_model(base_model, val_loader, "Base System", "base_cm")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Evaluating Base System (Frozen DistilBERT + Untrained Head)...


/tmp/ipykernel_4394/926489330.py:19: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}


In [104]:
# upgrading torchao due to having an older ver
!pip install --upgrade torchao

In [105]:
# LoRA Configuration
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"] # Targeting DistilBERT attention modules
)

adapted_model = get_peft_model(base_model, lora_config).to(device)
adapted_model.print_trainable_parameters()

# Training Loop
optimizer = torch.optim.AdamW(adapted_model.parameters(), lr=2e-4)
epochs = 3

print("\nTraining Adapted System with LoRA...")
adapted_model.train()
for epoch in range(epochs):
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = adapted_model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f}")

trainable params: 739,586 || all params: 67,694,596 || trainable%: 1.0925

Training Adapted System with LoRA...


/tmp/ipykernel_4394/926489330.py:19: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}


Epoch 1/3 | Loss: 0.1348
Epoch 2/3 | Loss: 0.0014
Epoch 3/3 | Loss: 0.0002


In [106]:
print("\nEvaluating Adapted System...")
adapted_metrics = evaluate_model(adapted_model, val_loader, "LoRA Adapted System", "lora_cm")

# Quantitative Comparison Table
results_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Base System': base_metrics,
    'LoRA Adapted': adapted_metrics
})

results_df.to_csv('results/metrics_table.csv', index=False)
print("\n--- Final Quantitative Comparison ---")
print(results_df.to_string(index=False))


Evaluating Adapted System...


/tmp/ipykernel_4394/926489330.py:19: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}



--- Final Quantitative Comparison ---
   Metric  Base System  LoRA Adapted
 Accuracy     0.685000           1.0
Precision     0.887755           1.0
   Recall     0.430693           1.0
 F1-Score     0.580000           1.0
